## Importacion de librerias

In [ ]:
import pandas as pd
import numpy as np
import io
import warnings
import re
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

print("Listo, librerias cargadas")

## Carga del DATASET

In [ ]:
import zipfile, io

with zipfile.ZipFile('/content/sample_data/steam_games.csv.zip', 'r') as z:
    with z.open('steam_games.csv') as f:
        df_raw = pd.read_csv(f)

print(f"Dataset cargado: {df_raw.shape[0]} filas y {df_raw.shape[1]} columnas")
print(f"Columnas disponibles: {df_raw.columns.tolist()}")
print(f"\nValores faltantes por columna:")
print(df_raw.isnull().sum())

csv_buffer = io.StringIO()
df_raw.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)

df_raw.head(3)

## Primer Agente

In [ ]:
class AgenteNormalizador:

    def __init__(self):
        self.encoders = {}
        self.scaler   = MinMaxScaler()
        self.acciones = []

    def limpiar_texto(self, df):
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].str.strip().str.lower()
        self.acciones.append("Texto puesto en minusculas y sin espacios extra")
        return df

    def sacar_outliers(self, df):
        raros = df['temperatura_c'].apply(lambda x: x < 0 or x > 60).sum()
        df['temperatura_c'] = df['temperatura_c'].apply(
            lambda x: np.nan if (x < 0 or x > 60) else x
        )
        self.acciones.append(f"Temperaturas imposibles eliminadas: {raros}")
        return df

    def rellenar_vacios(self, df):
        for col in df.select_dtypes(include=[np.number]).columns:
            vacios = df[col].isnull().sum()
            if vacios > 0:
                mediana = df[col].median()
                df[col] = df[col].fillna(mediana)
                self.acciones.append(f"Columna '{col}': {vacios} vacios rellenados con {mediana:.1f}")
        return df

    def convertir_texto_a_numero(self, df, columnas):
        for col in columnas:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            self.encoders[col] = le
        self.acciones.append(f"Columnas convertidas a numero: {columnas}")
        return df

    def escalar(self, df, columnas):
        df[columnas] = self.scaler.fit_transform(df[columnas])
        self.acciones.append(f"Numeros escalados entre 0 y 1: {columnas}")
        return df

    def ejecutar(self, csv_buffer):
        print("=" * 50)
        print("  AGENTE 1 - NORMALIZADOR")
        print("=" * 50)

        df = pd.read_csv(csv_buffer)
        print(f"Dataset recibido: {df.shape}")

        cols_texto  = ['genre', 'developer', 'types']
        cols_num    = ['original_price', 'discount_price', 'achievements']
        col_destino = 'tiene_descuento'


        def limpiar_precio(val):
            if pd.isna(val): return np.nan
            s_val = str(val).lower()
            if s_val == 'free': return 0.0
            numeros = re.sub(r'[^0-9.]', '', s_val)
            try:
                return float(numeros)
            except ValueError:
                return np.nan

        df['original_price']  = df['original_price'].apply(limpiar_precio)
        df['discount_price']  = df['discount_price'].apply(limpiar_precio)

        df['achievements'] = df['achievements'].fillna(0)

        df['tiene_descuento'] = df['discount_price'].notna().astype(int)

        cols_utiles = ['genre', 'developer', 'types',
                       'original_price', 'discount_price', 'achievements',
                       'tiene_descuento']
        df = df[cols_utiles].copy()

        print(f"Columnas seleccionadas para el modelo: {df.columns.tolist()}")

        df = self.limpiar_texto(df)
        df = self.rellenar_vacios(df)
        df = self.convertir_texto_a_numero(df, cols_texto)
        df = self.escalar(df, cols_num)

        print("\nCosas que hizo el agente:")
        for a in self.acciones:
            print(f"  - {a}")

        print(f"\nDataset limpio listo: {df.shape}")
        return df, col_destino

agente1 = AgenteNormalizador()
csv_buffer.seek(0)
df_limpio, target = agente1.ejecutar(csv_buffer)

print("\nPrimeras filas del dataset limpio:")
df_limpio.head()

## Segundo Agente

In [ ]:
class AgenteEntrenador:

    def __init__(self):
        self.modelos = {
            'Arbol de Decision' : DecisionTreeClassifier(max_depth=4, random_state=42),
            'KNN (vecinos)'     : KNeighborsClassifier(n_neighbors=5),
            'Naive Bayes'       : GaussianNB()
        }
        self.resultados   = {}
        self.mejor_modelo = None
        self.mejor_nombre = ''
        self.reporte      = ''

    def ejecutar(self, df, col_destino):
        print("=" * 50)
        print("  AGENTE 2 - ENTRENADOR")
        print("=" * 50)

        X = df.drop(columns=[col_destino]).values
        y = df[col_destino].values

        print(f"Columnas usadas como entrada: {X.shape[1]}")
        print(f"Total de ejemplos: {X.shape[0]}")
        print(f"Ventas bajas: {(y==0).sum()} | Ventas altas: {(y==1).sum()}")
        print()

        for nombre, modelo in self.modelos.items():
            puntuaciones = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
            self.resultados[nombre] = puntuaciones.mean()
            print(f"  {nombre:<22}  precision promedio: {puntuaciones.mean():.4f}")

        self.mejor_nombre = max(self.resultados, key=self.resultados.get)
        self.mejor_modelo = self.modelos[self.mejor_nombre]
        self.mejor_modelo.fit(X, y)

        y_pred = self.mejor_modelo.predict(X)
        self.reporte = classification_report(y, y_pred, target_names=['Baja', 'Alta'])

        print(f"\nModelo elegido: {self.mejor_nombre}")
        print(f"Precision: {self.resultados[self.mejor_nombre]:.4f}")
        print("Entrenamiento terminado.")

        metricas = {
            'nombre_modelo'  : self.mejor_nombre,
            'precision'      : self.resultados[self.mejor_nombre],
            'todos'          : self.resultados,
            'reporte'        : self.reporte,
            'n_ejemplos'     : X.shape[0],
            'n_columnas'     : X.shape[1]
        }
        return metricas, self.mejor_modelo

agente2 = AgenteEntrenador()
metricas, modelo_final = agente2.ejecutar(df_limpio, target)

## Api Key

In [ ]:
!pip install -q langchain-mistralai langgraph

import os
from getpass import getpass

os.environ["MISTRAL_API_KEY"] = getpass("Pega tu API Key de Mistral: ")

print("API key configurada")

## Tercer Agente

In [ ]:
class AgenteComunicador:

    def nivel_precision(self, valor):
        if valor >= 0.90:
            return "muy buena"
        elif valor >= 0.75:
            return "buena"
        elif valor >= 0.60:
            return "aceptable"
        else:
            return "baja, se recomienda revisar los datos"

    def ejecutar(self, metricas):
        print("=" * 50)
        print("  AGENTE 3 - COMUNICADOR")
        print("=" * 50)

        prec  = metricas['precision']
        nivel = self.nivel_precision(prec)
        todos = metricas['todos']

        ranking = sorted(todos.items(), key=lambda x: x[1], reverse=True)
        ranking_txt = "\n".join(
            [f"     {i+1}. {nom:<22} {val:.4f}" for i, (nom, val) in enumerate(ranking)]
        )

        reporte = f"""
================================================
   REPORTE FINAL DEL SISTEMA
   Analisis de Juegos - Steam
================================================

QUE SE HIZO:
  Se trabajó con {metricas['n_ejemplos']} registros de juegos de la
  plataforma Steam. El objetivo fue predecir si un juego tiene o
  no un precio con descuento aplicado.

  Se utilizaron {metricas['n_columnas']} variables predictoras:
  genero, desarrollador, tipo de juego, precio original,
  precio con descuento y cantidad de logros (achievements).

COMO SE PROCESO:
  Primero el Agente Normalizador limpio los datos: acomodo el
  texto, elimino valores imposibles, relleno los valores
  vacios y convirtio todo a numeros.

  Despues el Agente Entrenador probo tres modelos distintos:

{ranking_txt}

RESULTADO:
  El mejor modelo fue: {metricas['nombre_modelo']}
  Con una precision de {prec:.2%} — considerada {nivel}.

DETALLE POR CLASE:
{metricas['reporte']}
CONCLUSION:
  El modelo puede distinguir bastante bien si un juego tiene
  descuento o no. En un proyecto mas completo se podrian agregar
  mas datos como la fecha de lanzamiento, cantidad de resenas
  o el numero de jugadores activos.

Reporte generado por el Agente Comunicador.
================================================
"""
        print(reporte)
        return reporte


# Ejecutar
agente3 = AgenteComunicador()
reporte_final = agente3.ejecutar(metricas)